# Image Pre-Processing

### Goal: Maintain RGB values and re-size to consistent 128-128 images

In [10]:
#libraries

from PIL import Image
import numpy as np
import os
import pandas as pd

In [13]:
#Function to convert and resize images

def convert_images(input_dir, output_dir, output_csv, size=(128, 128), label=None):
    #Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    #Supported image formats
    supported = {".jpg", ".jpeg", ".png"}
    
    data = []
    for filename in os.listdir(input_dir):
        ext = os.path.splitext(filename)[1].lower()
        #Check if file is supported type
        if ext not in supported:
            print(f"Skipping unsupported file: {filename}")
            continue
        
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        with Image.open(input_path) as img:
            #Resize the image to the given dimensions using a high-quality method
            #COnsistent treament for RGB values
            img = img.convert("RGB")  
            img = img.resize(size, Image.LANCZOS)
            arr = np.array(img).astype(np.float32) / 255.0  # normalize [0, 1]
            flat = arr.flatten()
            
            img.save(output_path)
            # normalize
            img = np.array(img).astype(np.float32) / 255.0
        data.append([filename + str(label), label] + flat.tolist())
        print(f"Processed: {filename}")

    n_pixels = size[0] * size[1]
    col_names = ["filename", "label"] + [f"{c}_{i}" for i in range(n_pixels) for c in ("r", "g", "b")]

    df = pd.DataFrame(data, columns=col_names)
    #df.to_csv(output_csv, index=False)

    print(f"\nDone. Shape: {df.shape}")
    print(f"Label counts:\n{df['label'].value_counts()}")
    print(f"Saved to: {output_csv}")

    return df




In [ ]:
#Apply the function to the clean and contaminated image folders
clean = convert_images(input_dir="Datasets/clean", output_dir="resized_datasets/clean", output_csv="features_clean.csv", label=0)
contaminated = convert_images(input_dir="Datasets/contaminated", output_dir="resized_datasets/contaminated", output_csv="features_contaminated.csv", label=1)

df = pd.concat([
    clean,
    contaminated
]).reset_index(drop=True)
#file too big without compression, so saving as gzip
df.to_csv("features_merged.csv.gz", index=False, compression="gzip")


Processed: img_1.jpeg
Processed: img_10.jpeg
Processed: img_100.jpg
Processed: img_1000.jpeg
Processed: img_1001.jpeg
Processed: img_1002.jpg
Processed: img_1003.jpeg
Processed: img_1004.jpeg
Processed: img_1005.jpg
Processed: img_1006.jpeg
Processed: img_1007.jpeg
Processed: img_1008.jpeg
Processed: img_1009.jpeg
Processed: img_101.jpeg
Processed: img_1010.jpeg
Processed: img_1011.jpeg
Processed: img_1012.jpeg
Processed: img_1013.jpeg
Processed: img_1014.jpeg
Processed: img_1015.jpeg
Processed: img_1016.jpeg
Processed: img_1017.jpg
Processed: img_1018.jpeg
Processed: img_1019.jpeg
Processed: img_102.jpg
Processed: img_1020.png
Processed: img_1021.jpeg
Processed: img_1022.jpeg
Processed: img_1023.jpeg
Processed: img_1024.jpg
Processed: img_1025.jpeg
Processed: img_1026.jpg
Processed: img_1027.jpeg
Processed: img_1028.jpeg
Processed: img_1029.jpg
Processed: img_103.jpeg
Processed: img_1030.jpeg
Processed: img_1031.jpeg
Processed: img_1032.jpeg
Processed: img_1033.jpeg
Processed: img_103

In [17]:
print(len(df["filename"]), len(df["filename"].unique()))

missing = df.isnull().sum()
print(missing[missing > 0])

500 500
Series([], dtype: int64)
